# Heel-video shoe tracker — SAM 2 validation

This notebook answers one question cheaply: **when you twist your foot to show the shoe, does SAM 2 keep a clean outline on the shoe the whole time?**

If it does, replacing the shoe by compositing is realistic. If the outline slips or falls apart, the twist beat needs the heavier 3D approach.

**Two rules that make or break this test**
1. **Initialize on the TWIST frame, not the walk-in.** Pick a frame where your foot is planted, still, and turned to show the heel. The blurry walk-in/out frames are the hardest for SAM 2 — start on a clean one and let it track outward from there.
2. **Box the shoe, don't click the leg.** A tight rectangle around just the shoe is far more reliable than a single point (a point near the ankle grabs the whole leg).

**How to run it**
1. `Runtime` -> `Change runtime type` -> **GPU** -> Save.
2. Optional but recommended: `File` -> `Save a copy in Drive`, so your edits stick (the "saving failed" banner just means this copy is read-only).
3. Run top to bottom. Upload one short clip, pick the twist frame, box the shoe, watch the preview.

**If you hit an out-of-memory error:** it's Colab's free GPU, usually from re-running the setup cell many times. `Runtime` -> `Restart session`, then run top to bottom once. Cell 4 already downscales frames and Cell 6 offloads to CPU to keep memory low; if it still happens, drop `scale=720` lower or switch to the smaller model (`sam2.1_hiera_small.pt` / `sam2.1_hiera_s.yaml`).

## 1. Confirm you have a GPU
If this shows nothing, set the runtime to GPU (see above).

In [ ]:
!nvidia-smi

## 2. Install SAM 2 and download the model
A couple of minutes the first time.

In [ ]:
!git clone https://github.com/facebookresearch/sam2.git
%cd sam2
!pip install -q -e .
!mkdir -p checkpoints
!wget -q -P checkpoints https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt
print("Done. Model downloaded.")

## 3. Upload one of your clips
Run, then pick a video. Keep the first test short.

In [ ]:
from google.colab import files
uploaded = files.upload()
VIDEO_PATH = "/content/sam2/" + list(uploaded.keys())[0]
print("Uploaded:", VIDEO_PATH)

## 4. Break the video into frames

Downscaled to 720px wide at 12 fps to keep Colab's free GPU from running out of memory. If you hit an out-of-memory error, lower `scale` (e.g. 540) or trim to the relevant seconds by adding `-ss 2 -t 6` (6 seconds from the 2s mark).

In [ ]:
import os, glob
FRAME_DIR = "/content/frames"
os.system(f"rm -rf {FRAME_DIR}")
os.makedirs(FRAME_DIR, exist_ok=True)
# scale=720:-2 keeps aspect ratio; fps=12 thins frames -> much less memory
os.system(f'ffmpeg -loglevel error -i "{VIDEO_PATH}" -vf "scale=720:-2,fps=12" '
          f'-q:v 3 -start_number 0 "{FRAME_DIR}/%05d.jpg"')
frames = sorted(glob.glob(f"{FRAME_DIR}/*.jpg"))
print(f"Extracted {len(frames)} frames (720px wide, 12 fps).")

## 5. Pick the TWIST frame

Set `INIT_FRAME` to a frame index where your foot is **planted and turned to show the shoe** — still and sharp, not the blurry walk-in. Re-run to scrub to different frames until you find a good one. Note the x,y grid — you'll box the shoe next.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

INIT_FRAME = 0          # <-- change this to your twist frame (0 .. total-1)

print("total frames:", len(frames), "-> valid INIT_FRAME is 0 to", len(frames) - 1)
img = Image.open(frames[INIT_FRAME]); w, h = img.size
fig, ax = plt.subplots(figsize=(11, 11 * h / w))
ax.imshow(img)
ax.set_xticks(range(0, w, max(1, w // 20)))
ax.set_yticks(range(0, h, max(1, h // 20)))
ax.grid(color="cyan", linestyle=":", linewidth=0.6, alpha=0.7)
ax.set_title(f"frame {INIT_FRAME} of {len(frames)}  —  {w} wide x {h} tall")
plt.show()

## 6. Box each shoe and preview the masks

List one box per shoe in `SHOE_BOXES` — a rectangle around **just that shoe**, read off the grid: `[x_left, y_top, x_right, y_bottom]`. One box for one shoe, two boxes when both feet are in frame. Each shoe becomes its own tracked object (shoe 1 = red, shoe 2 = green, ...).

Pick this frame so **every shoe you box is clearly visible** in it. Re-run and adjust until each mask covers its shoe and nothing else.

In [ ]:
import numpy as np
import torch
from sam2.build_sam import build_sam2_video_predictor

# ---- one box per shoe: [x_left, y_top, x_right, y_bottom] ----
SHOE_BOXES = [
    [400, 900, 700, 1200],       # shoe 1
    # [750, 900, 1050, 1200],    # shoe 2  <- uncomment when both feet are in frame
]
# --------------------------------------------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"
predictor = build_sam2_video_predictor(
    "configs/sam2.1/sam2.1_hiera_l.yaml",
    "checkpoints/sam2.1_hiera_large.pt",
    device=device,
)

state = predictor.init_state(video_path=FRAME_DIR,
                             offload_video_to_cpu=True, offload_state_to_cpu=True)
predictor.reset_state(state)

# add each shoe as its own object; obj_id = 1, 2, ...
for obj_id, box in enumerate(SHOE_BOXES, start=1):
    _, obj_ids, mask_logits = predictor.add_new_points_or_box(
        inference_state=state, frame_idx=INIT_FRAME, obj_id=obj_id,
        box=np.array(box, dtype=np.float32))

# the last call returns masks for ALL objects on this frame, in obj_ids order
init_masks = {oid: (mask_logits[k] > 0).cpu().numpy().squeeze()
              for k, oid in enumerate(obj_ids)}

palette = {1: [1, 0, 0], 2: [0, 1, 0], 3: [0, 0, 1], 4: [1, 1, 0]}
img = Image.open(frames[INIT_FRAME]); w, h = img.size
fig, ax = plt.subplots(figsize=(11, 11 * h / w))
ax.imshow(img)
for oid, m in init_masks.items():
    ov = np.zeros((*m.shape, 4)); ov[m] = palette.get(oid, [1, 0, 1]) + [0.5]
    ax.imshow(ov)
for oid, box in enumerate(SHOE_BOXES, start=1):
    ax.add_patch(plt.Rectangle((box[0], box[1]), box[2] - box[0], box[3] - box[1],
                               fill=False, edgecolor="yellow", linewidth=2))
ax.set_title("Each color = one shoe. Adjust the boxes until each covers its shoe only.")
plt.show()

## 7. Track every shoe through the whole clip

Tracks **outward in both directions** from your chosen frame — forward to the walk-out, backward to the walk-in — for all shoes at once.

In [ ]:
masks = {}   # frame index -> {obj_id: boolean mask}
def collect(reverse):
    for fi, oids, ml in predictor.propagate_in_video(
            state, start_frame_idx=INIT_FRAME, reverse=reverse):
        per_obj = {oid: (ml[k] > 0).cpu().numpy().squeeze() for k, oid in enumerate(oids)}
        masks.setdefault(fi, {}).update(per_obj)
collect(reverse=False)
collect(reverse=True)
print(f"Tracked {len(SHOE_BOXES)} shoe(s) across {len(masks)} frames.")

## 8. Build the preview video

Tints each shoe its color in every frame and stitches an MP4. **Watch the twist**: each color should stay glued to its shoe as the foot rotates.

In [ ]:
import cv2

bgr = {1: (0, 0, 255), 2: (0, 255, 0), 3: (255, 0, 0), 4: (0, 255, 255)}  # BGR
out_dir = "/content/overlay_frames"
os.system(f"rm -rf {out_dir}"); os.makedirs(out_dir, exist_ok=True)
for i, fpath in enumerate(frames):
    frame = cv2.imread(fpath)
    for oid, m in masks.get(i, {}).items():
        if m.any():
            tint = np.zeros_like(frame); tint[:] = bgr.get(oid, (255, 0, 255))
            frame = np.where(m[..., None], (0.5 * frame + 0.5 * tint).astype(np.uint8), frame)
    cv2.imwrite(f"{out_dir}/{i:05d}.jpg", frame)
os.system(f'ffmpeg -loglevel error -y -framerate 24 -i "{out_dir}/%05d.jpg" '
          f'-c:v libx264 -pix_fmt yuv420p /content/preview.mp4')
print("Wrote /content/preview.mp4")

In [ ]:
from IPython.display import HTML
from base64 import b64encode
data = b64encode(open("/content/preview.mp4", "rb").read()).decode()
HTML(f'<video width=500 controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

## How to read the result

- **Red stays locked on the shoe through the twist** -> compositing is viable; that red region is exactly where a new shoe would be painted in.
- **Red slips onto the leg/floor, shrinks, or flickers during the twist** -> that beat likely needs the 3D-rendering approach; the walk-in/out can still use this.

If tracking drifts, go back to cell 5 (try a different twist frame), then cell 6 (tighten the box, add a negative point on the leg), and re-run 6 -> 7 -> 8. Tell me what the twist looks like and we'll pick the build path.